# Falcon launch table web scraping

Applied Data Science Capstone. Rodrigo Ariel Quilodran.

Historical IBM course datasets, analysis executed 7 November 2026. AI-assisted preparation; source data and limitations are disclosed.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA, OUT, FIG = ROOT/'data', ROOT/'results', ROOT/'figures'
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})


## Historical Wikipedia page
Source: https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922 . The downloaded HTML is archived locally for reproducibility.

In [2]:
from bs4 import BeautifulSoup
import re
soup = BeautifulSoup((DATA/'wikipedia.html').read_text(),'html.parser')
print(soup.title)
tables = soup.select('table.wikitable')
print('Candidate launch tables:',len(tables))

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>
Candidate launch tables: 13


## Extract actual launch records
Parse numbered table rows and retain nine table cells. Remove footnotes and normalize whitespace. Date, booster, payload and landing outcome remain separate fields. This source has a later cutoff than the 90-row modeling snapshot.

In [3]:
rows=[]
for table in tables:
    for tr in table.find_all('tr'):
        th=tr.find('th')
        if th is None: continue
        number=th.get_text(' ',strip=True)
        cells=tr.find_all('td',recursive=False)
        if not number.isdigit() or len(cells)<9: continue
        for cell in cells:
            for sup in cell.find_all('sup'): sup.decompose()
        values=[re.sub(r'\s+',' ',c.get_text(' ',strip=True)) for c in cells[:9]]
        rows.append([int(number)]+values)
columns=['FlightNumber','DateTime','BoosterVersion','LaunchSite','Payload','PayloadMass','Orbit','Customer','LaunchOutcome','BoosterLanding']
scraped=pd.DataFrame(rows,columns=columns).drop_duplicates()
assert len(scraped)>100
scraped.to_csv(OUT/'scraped_launches.csv',index=False)
print('Extracted records:',len(scraped))
scraped.head(10)

Extracted records: 121


,FlightNumber,DateTime,BoosterVersion,LaunchSite,Payload,PayloadMass,Orbit,Customer,LaunchOutcome,BoosterLanding
0,1,"4 June 2010, 18:45",F9 v1.0 B0003.1,"CCAFS , SLC-40",Dragon Spacecraft Qualification Unit,,LEO,SpaceX,Success,Failure (parachute)
1,2,"8 December 2010, 15:43",F9 v1.0 B0004.1,"CCAFS , SLC-40",Dragon demo flight C1 (Dragon C101),,LEO ( ISS ),NASA ( COTS ) NRO,Success,Failure (parachute)
2,3,"22 May 2012, 07:44",F9 v1.0 B0005.1,"CCAFS , SLC-40",Dragon demo flight C2+ (Dragon C102),"525 kg (1,157 lb)",LEO ( ISS ),NASA ( COTS ),Success,No attempt
3,4,"8 October 2012, 00:35",F9 v1.0 B0006.1,"CCAFS , SLC-40",SpaceX CRS-1 (Dragon C103),"4,700 kg (10,400 lb)",LEO ( ISS ),NASA ( CRS ),Success,No attempt
4,5,"1 March 2013, 15:10",F9 v1.0 B0007.1,"CCAFS , SLC-40",SpaceX CRS-2 (Dragon C104),"4,877 kg (10,752 lb)",LEO ( ISS ),NASA ( CRS ),Success,No attempt
5,6,"29 September 2013, 16:00",F9 v1.1 B1003,"VAFB , SLC-4E",CASSIOPE,"500 kg (1,100 lb)",Polar orbit LEO,MDA,Success,Uncontrolled (ocean)
6,7,"3 December 2013, 22:41",F9 v1.1 B1004,"CCAFS , SLC-40",SES-8,"3,170 kg (6,990 lb)",GTO,SES,Success,No attempt
7,8,"6 January 2014, 22:06",F9 v1.1,"CCAFS , SLC-40",Thaicom 6,"3,325 kg (7,330 lb)",GTO,Thaicom,Success,No attempt
8,9,"18 April 2014, 19:25",F9 v1.1,"Cape Canaveral , LC-40",SpaceX CRS-3 (Dragon C105),"2,296 kg (5,062 lb)",LEO ( ISS ),NASA ( CRS ),Success,Controlled (ocean)
9,10,"14 July 2014, 15:15",F9 v1.1,"Cape Canaveral , LC-40",Orbcomm-OG2 -1 (6 satellites),"1,316 kg (2,901 lb)",LEO,Orbcomm,Success,Controlled (ocean)
